In [36]:
pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [37]:
%pip install langchain langchain-community langchain-ollama langchain-text-splitters langchain-chroma chromadb -q

Note: you may need to restart the kernel to use updated packages.


In [38]:
import os
os.makedirs("docs",exist_ok=True)
#document-1: company Overview
doc1 = """
TechNova Company Overview

TechNova was founded in 2018 by Sarah Chen and Marcus Patel in Austin, Texas.
The company specializes in AI-powered inventory management systems for retail businesses.
TechNova currently serves over 500 clients across North America and Europe.
Their flagship product, NovaStock, uses machine learning to predict demand and automate reordering.
In 2023, TechNova raised $40 million in Series B funding led by Sequoia Capital.
"""
#document-2 product features
doc2 = """
NovaStock Product Features

NovaStock is TechNova's core product. Key features include:
- Real-time inventory tracking across multiple warehouse locations
- AI-driven demand forecasting with 94% accuracy
- Automated purchase order generation when stock falls below threshold
- Integration with Shopify, WooCommerce, and SAP
- Mobile app available on iOS and Android
- Pricing starts at $299/month for small businesses
"""
# Document 3: Support Policy
doc3 = """
TechNova Customer Support Policy

TechNova offers three tiers of customer support:
1. Starter: Email support with 48-hour response time (included in all plans)
2. Pro: Live chat + email with 4-hour response time ($99/month add-on)
3. Enterprise: 24/7 dedicated support manager + phone + email (custom pricing)

All customers get access to the self-service knowledge base and video tutorials.
Onboarding sessions are available for Pro and Enterprise customers.
"""
with open("docs/company_overview.txt","w") as f:
    f.write(doc1)
with open("docs/product_features.txt","w") as f:
    f.write(doc2)
with open("docs/support_policy.txt","w") as f:
    f.write(doc3)
print("Sample documents created in ./docs/")
    


Sample documents created in ./docs/


In [39]:
from IPython.display import HTML, display

display(HTML("""
<h2 style="color:green;">STEP-2: Load Documents</h2>

<p style="color:red;">
We use LangChain's <span style="color:GREEN;">DirectoryLoader</span>
to read all .txt files from our folder.<br>

LangChain has loaders for PDFs, webpages, Notion, Google Docs,
YouTube transcripts, and more — the API is the same;
only the loader class changes.
<p style="color:Brown">Each Loaded File Becomes An Document Object With
<ul>
<li>Page_Content-the actual text</li>
<li>metadata-info like the source file path</li>
</ul>

</p>
</p>


"""))

In [40]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader,PyPDFLoader
loader=DirectoryLoader(
    path=r"C:\Users\Mounisree\Downloads",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)
documents=loader.load()
print(f"Loaded {len(documents)} documents")
print("\n---Preview of first Document---")
print(f"Source:{documents[0].metadata['source']}")
print(f"Content Review:\n{documents[0].page_content[:200]}...")


Loaded 174 documents

---Preview of first Document---
Source:C:\Users\Mounisree\Downloads\10-th Marklist.pdf
Content Review:
Digitally	signed	on	
25/10/2024	20:23:39	IST
BOARD	OF	SECONDARY	EDUCATION
ANDHRA	PRADESH,	INDIA
SECONDARY	SCHOOL	CERTIFICATE
REGULAR
Certified	that
BATTA	MOUNISREE
Father's	Name
BATTA	RAGHURAMAIAH
Mot...


In [41]:
from IPython.display import HTML, display

display(HTML("""
<h2 style="color:green">STEP-3 Split Documents into Chunks</h2>
<p> why Split ?Because:
<ul>
<li>LLMs have a context window limit -you can't fed the entire book at once</li>
<li>smaller,focused chunks give more precise retrieval results</li>
<li>you want the retrieved chunk to be just the relievent section,not entire document</li>
</ul>
we use RecursiveCharacterSplitter -it is an langchain splitter it tries to split on paragraphs first,then sentences,then words,to keep chunks meaningful
<h4>key parameters</h4>
<ul>
<li><h5>chunk_size</h5>-max characters per chunk(500 is good for short docs)</li>
<li><h5>chunk_overlap</h5>-how many characters overlap between consecutive chunks (prevents losing context at boundaries)</li>
<li><h5>length_function</h5>tells the splitter how to measure the size of the chunk here if the size of current chunk size greater then the chunk_size then we further split() 
if length(chunk) > chunk_size:
    split_further()
</li>
<li>split_documents()--these method takes the larger documents objects and return smaller document objects </li>
<ul>
<li><h5>Example</h5>
[
  Document(
    page_content="ABCDEFGHIJKLMNOPQRSTUVWXYZ"
  )
]
</li>
<li>
chunk_size=10,
overlap=2
</li>
<li>chunks=splitter.split_documents(documents)</li>
<li>[
  Document("ABCDEFGHIJ"),
  Document("IJKLMNOPQR"),
  Document("QRSTUVWXYZ")
]</li>
</ul>

</ul>
and here check the overlap between each document is 2 characters only
</p>

"""))

In [42]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,#each chunk is at most 500 characters
    chunk_overlap=50, # 50-char overlap between chunks to preserve context
    length_function=len
)
chunks=splitter.split_documents(documents)
print(f"split {len(documents)} documents into {len(chunks)} chunks")
print("---Example chunk---")
print(f"Source:{chunks[0].metadata['source']}")
print(f"Length:{len(chunks[0].page_content)} chars")
print(f"Content:{chunks[0].page_content}")
    

split 174 documents into 575 chunks
---Example chunk---
Source:C:\Users\Mounisree\Downloads\10-th Marklist.pdf
Length:458 chars
Content:Digitally	signed	on	
25/10/2024	20:23:39	IST
BOARD	OF	SECONDARY	EDUCATION
ANDHRA	PRADESH,	INDIA
SECONDARY	SCHOOL	CERTIFICATE
REGULAR
Certified	that
BATTA	MOUNISREE
Father's	Name
BATTA	RAGHURAMAIAH
Mother's	Name
BATTA	JAYALAKSHMI
Roll	No.
2117133762
Date	of	Birth
25/07/2006
School	Name
VSSC	HOLY	CROSS	HIGH	SCHOOL,	SULLURPETA,	NELLORE
has	Registered	and	Passed	SSC	Examination	held	in
	JUNE	
2021
	in	
FIRST
	Division	with	
ENGLISH	
as	medium	of	instruction.


In [43]:
from IPython.display import HTML, display

display(HTML("""
<h2> STEP 4-Create Embeddings + Store in Vector DB</h2>
this is the heart of RAG

<h3>What Are Embeddings ?</h3>
<p>An Embedding model converts text into a list of numbers known as vectors that captures the semantic relationship and meaning of the text</p>
<ul>
<li>Texts with similar meaning-vectors that are close together in space</li>
<li>Texts with different meaning-vectors that are far apart</li>
 <h4>Example:</h4>
 <li>"What is the price of NovaStock?" and "Pricing starts at $299/month" → close vectors ✅</li>
<li>"What is the price of NovaStock?" and "TechNova was founded in Austin" → far vectors ✅</li>
</ul>
<h3>What is ChromaDB?</h3>
<p>ChromaDB is a vector database--it stores your chunk vectors and quickly find the most similar ones to a query vector 
We are using <span style="color:RED">nomic-embed-text</span> via ollama-- A great open-source embedding model that runs fully locally

</p>
"""))

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
print("genarting the emdeddings with nomic-embed-text")
embedding_model=OllamaEmbeddings(model="nomic-embed-text")
# Create a ChromaDB vector store and populate it with our chunks + their embeddings
# persist_directory="./chroma_db" saves the index to disk so we don't re-embed on restart
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
print(f"Stored {len(chunks)} chunks in chromaDB")
print("vector is saved to ./chroma_db")

genarting the emdeddings with nomic-embed-text


In [ ]:
from IPython.display import HTML,display
display(HTML(""" 
<h2>Step 5 --Set Up the Retriever</h2>
<p>A retriever is the component that given a users question,fetches the most relavent chunks from the vector store</p>
<h5>under the hood it:</h5>
<ul>
<li>Embeds the question using the same embedding model</li>
<li>Does a cosine similarity search in the vector space</li>
<li>returns the top-k most similar chunks</li>
<li>if k=3 means retrieves the 3 best matching chunks for every question</li>
</ul>
"""))

In [ ]:
# Create a retriever from the vector store
# search_kwargs={"k": 3} → retrieve top 3 most relevant chunks
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
    )
#quick test-lets see what gets retrieved for sample question
test_query="How much does Novastock cost?"
retrieved_docs=retriever.invoke(test_query)
print(f"Query:'{test_query}'")
print(f"Retrieved{len(retrieved_docs)} chunks")
for i,doc in enumerate(retrieved_docs):
    print(f"-----Chunk{i+1} (from {doc.metadata['source']})---")
    print(doc.page_content)
    print()
    

In [ ]:
from IPython.display import HTML,display
display(HTML("""
<h2>STEP 6-Load the LLM (via ollama)</h2>
<p>Now we load the language model that will read the retrieved context and genarate answer
we are using <span style="color:red">llama3.1:latest</span>-Metas powerful open-source model,running 100% locally via ollama
<h4>Parameters:</h4>
<ul>
<li>Temparature=0 --temparature parameters in llm controls the probability ,randomness,genaralization and creativity of answers of an llm usually it starts with 0 and go on upto 2
</li>
<li>Low-Temparature=safer and more predicted text</li>
<li>High-Temparature=more creative and varied output</li>
</ul>
You can swap llama3.1:latest for<span style="color:green"> mistral,gemma2,phi3</span> or any model you have pulled in ollama 
</p>
"""))

In [ ]:
from langchain_ollama import ChatOllama
llm=ChatOllama(
    model="llama3.1:latest",
    temparature=0
)
response=llm.invoke("what is TechNova's flagship Product?")
print("LLM Answer without any context:")
print(response.content)

In [ ]:
from IPython.display import HTML,display
display(HTML("""
<h2>Step-7 Build The Prompt Template</h2>
<p>
The <span style="color:red">Prompt template</span> is the instructions we give the llm
<ul>
<li>here is some context(the retrived chunks)</li>
<li>here is the question</li>
<li>Answer only based on context-don't make stff up</li>
</ul>
These is the critical Part that keeps the LLM <span style="color:GREEN">Grounded</span> in your documents
<h5>Parameters:</h5>
<ul>
<li>input_variables=["context","question"]</li>
<li>
# The {context} placeholder will be filled with retrieved chunks
</li>
<li># The {question} placeholder will be filled with the user's question
</li>
<li>template=defines how the model should behave </li>

</ul>
</p>
"""))

In [ ]:
from langchain_core.prompts import PromptTemplate
# The {context} placeholder will be filled with retrieved chunks
# The {question} placeholder will be filled with the user's question
prompt_template=PromptTemplate(
    input_variables=["context","question"],
    template="""You are a helpful assistant. Answer the question using ONLY the information 
provided in the context below. If the answer is not in the context, say 
"I don't have that information in my knowledge base."

Context:
{context}

Question: {question}

Answer:"""
)
print("Prompt Template Created")
print("Template Preview")
print(prompt_template.template)

    




In [ ]:
from IPython.display import HTML,display
display(HTML("""
<h2>STEP 8--Assemble the RAG Chain</h2>
<p>Now we put it all together into a chain</p>
<h4>LangChain's LCEL(LangChain Expression Language)</h4>
<p>Uses the | pipe Operator-just like unix pipes to connect Componext</p>
<h4>Heres the Flow</h4>
<h3>
Here's the flow:</h3>
<div style="white-space: pre; font-family: monospace;">
Here's the flow:

User Question
     │
     ├──► Retriever ──► fetches relevant chunks ──► {context}
     │
     └──► passthrough ──────────────────────────► {question}
                                                       │
                                                       ▼
                                               Prompt Template
                                                       │
                                                       ▼
                                                      LLM
                                                       │
                                                       ▼
                                               String Output
</div>
<h4>the pre in div is <pre> preserves spaces and line breaks exactly as written.</h4>
The | operator means:

"Take the output from the left side and pass it as input to the right side."
"""))

In [ ]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.output_parsers import StrOutputParser
def format_docs(docs):
    """join retieved document chunks into a single context sgtring """
    return "\n\n".join(doc.page_content for doc in docs)
# Build the RAG chain using LCEL (LangChain Expression Language)
rag_chain=(
    RunnableParallel({
        "context":retriever|format_docs,
        "question":RunnablePassthrough()
        })
    | prompt_template
    |llm
    |StrOutputParser()
    )
print("RAG chain assembled!")
print("Flow:Question->Retriver->prompt->llm->Answer")

In [25]:
from IPython.display import HTML,display
display(HTML("""
<p>Lets ask the questions that LLM Cound never Answer from memory but answer with out Documents
<br></br>
Watch how it uses the retrieved chunks to give accurate,grounded answers
</p>
"""))

In [ ]:
def ask(question):
    print(f"\n{'='*60}")
    print(f"Question:{question}")
    print(f"{'='*60}")

    retrieved=retriever.invoke(question)
    print(f"\n Retrieved {len(retrieved)} chunks")
    for i,doc in enumerate(retrieved):
        src=doc.metadata['source'].split('/')
        print(f" [{i+1}] {src}:{doc.page_content[:80]}")
    answer=rag_chain.invoke(question)
    print(f"Answer:{answer}")
    print()  
    

In [ ]:
ask("Who founded TechNova and when?")